# scarlatti-doodle project!

## Zip 파일 가상 공간에 마운트
#### this code block was written using AI

In [31]:
import io
import os
import zipfile
from music21 import converter, midi, environment

# music21 경로 억까 방지 환경 설정
try:
    environment.Environment()['directoryScratch'] = '/tmp'
except:
    pass

def mount_scarlatti():
    """scarlatti.zip을 가상 메모리에 마운트하고, 
    파일을 언제든 꺼내 쓸 수 있는 가상 폴더 객체(ZipFile)와 파일 명단을 반환합니다."""
    ZIP_FILE_PATH = "original_midi.zip"
    
    if not os.path.exists(ZIP_FILE_PATH):
        raise FileNotFoundError(f"⚠️ '{ZIP_FILE_PATH}' 파일이 없습니다. 왼쪽 탐색기에 업로드해 주세요!")
        
    print("🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...")
    
    # 파일을 RAM으로 읽어오기
    with open(ZIP_FILE_PATH, "rb") as f:
        zip_buffer = io.BytesIO(f.read())
        
    # Zip 아카이브 객체 생성
    archive = zipfile.ZipFile(zip_buffer)
    
    # 미디 파일 목록만 싹 긁어오기
    midi_files = [f for f in archive.namelist() if f.lower().endswith(('.mid', '.midi'))]
    print(f"📦 마운트 완료! 총 {len(midi_files)}개의 가상 미디 폴더 준비 완료.")
    
    return archive, midi_files

def load_midi_from_virtual_folder(archive, file_path):
    """가상 폴더(archive)와 파일 경로를 주면, 디스크 IO 없이 music21 Score 객체로 즉시 변환합니다."""
    midi_raw_bytes = archive.read(file_path)
    
    # 샌드박스 바이트 스트림 파싱 (우리가 방금 성공한 그 정석 코드)
    mf = midi.MidiFile()
    mf.readstr(midi_raw_bytes)
    return midi.translate.midiFileToStream(mf)

# 🔥 [실행] 가상 폴더 연결하기
# 앞으로 코드 짤 때는 이 두 변수(scarlatti_folder, file_list)만 가지고 노시면 됩니다!
scarlatti_folder, file_list = mount_scarlatti()
print("앞으로 scarlatti_folder, file_list에 접근해서 사용! load_midi_from_virtual_folder 함수 사용")

# 미디 시각화하는 함수 작성
from music21 import converter
import matplotlib.pyplot as plt
import numpy as np # 숫자 변환을 도와주는 도구

def ShowMidi(score):
    # 1. 데이터를 뽑을 때 아예 실수(float)로 강제 변환합니다.
    raw_pitches = [float(n.pitch.ps) for n in score.flatten().notes if getattr(n, 'isNote', False) and not n.isChord]
    raw_offsets = [float(n.offset) for n in score.flatten().notes if getattr(n, 'isNote', False) and not n.isChord]

    # 2. 혹시나 섞여 있을지 모르는 '이상한 값'들을 넘파이(numpy)로 한 번 더 걸러냅니다.
    pitches = np.array(raw_pitches, dtype=float)
    offsets = np.array(raw_offsets, dtype=float)

    # 3. 그리기
    plt.figure(figsize=(12, 4))
    plt.scatter(offsets, pitches, s=2, c='orange')
    plt.show()

    score.write()

print("ShowMidi 함수 준비됨!")

🛸 가상 메모리(RAM)에 스칼라티 555개 소나타 마운트 중...
📦 마운트 완료! 총 555개의 가상 미디 폴더 준비 완료.
앞으로 scarlatti_folder, file_list에 접근해서 사용! load_midi_from_virtual_folder 함수 사용
ShowMidi 함수 준비됨!


#### 전처리 함수들 준비

In [34]:
from music21 import *
import re


def Score_TrebleBassSeparation(inputScore: stream.Score, splitPoint: str ="C4"):
    '''최신 music21 버전에 맞춘 오차 없는 높은음/낮은음자리 분리 함수'''
    
    treble = stream.Part()
    bass = stream.Part()
    outputScore = stream.Score()

    treble.append(clef.TrebleClef())
    bass.append(clef.BassClef())

    splitPointPitch = pitch.Pitch(splitPoint)

    for n in inputScore.recurse().notes:
        
        # 1. 대악보 기준의 절대 박자 위치를 정확하게 구합니다.
        offset = n.getOffsetInHierarchy(inputScore)

        # 2. 최신 버전에 맞춰 안전하게 화음을 단음 리스트로 분리합니다.
        notesToProcess = n.notes if n.isChord else [n]

        for singleNote in notesToProcess:
            # 💡 [핵심 해결책] 새 음표를 만들 때, 원본(n)이 가지고 있던 정밀한 duration을 그대로 주입합니다.
            clean_note = note.Note(pitch=singleNote.pitch, duration=n.duration)

            # 3. 순수 Pitch 비교 후 각각의 파트에 정확한 절대 위치(offset)로 insert!
            if clean_note.pitch < splitPointPitch:
                bass.insert(offset, clean_note)
            else:
                treble.insert(offset, clean_note)

    outputScore.append(treble)
    outputScore.append(bass)

    return outputScore

def Score_TransposeToAllKeys(inputScore):
    '''스코어 파일을 모든 조로 전조해서 리턴'''
    pass

def Score_SliceByMeasures(inputScore):
    '''스코어 파일을 특정 마디 길이만큼 나눠서 리턴'''
    pass


def Score_MaskNotes(inputScore):
    '''멜로디 데이터에서 일부 데이터들 마스킹해서 리턴'''
    pass

def ScoreToDataset(inputScore):
    '''스코어 파일을 ai 학습용 데이터셋으로 변환해서 리턴, 코드 정보와 박자정보 추가'''
    pass

    


In [35]:
score = converter.parse("sonatas_k-531_(c)sankey.mid")

trebble = Score_TrebleBassSeparation(score)

trebble.write('midi', fp="trebble.mid")



'trebble.mid'